In [1]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from torch.optim import Adam
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

In [2]:
BASE_DIR = r"C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR = f"{BASE_DIR}/classification/val"
TEST_DIR = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(
        degrees=0,
        scale=(0.8, 1.2)
    ),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=val_test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [5]:
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\SAKTHI/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:05<00:00, 3.68MB/s]


In [7]:
for parameter in model.features.parameters():
    parameter.requires_grad = False

input_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.BatchNorm1d(input_features),
    nn.Linear(input_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, len(train_dataset.classes))
)

model = model.to(device)

In [8]:
criterion = nn.CrossEntropyLoss()

optimizer = Adam(
    filter(lambda parameter: parameter.requires_grad, model.parameters())
)

In [9]:
BEST_MODEL_PATH = "models/efficientnetb0_best.pth"
PATIENCE = 5

In [10]:
EPOCHS = 25

history = {
    "accuracy": [],
    "val_accuracy": [],
    "loss": [],
    "val_loss": []
}

best_val_accuracy = 0.0
epochs_without_improvement = 0
best_model_state = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total
    val_loss /= val_total
    val_accuracy = val_correct / val_total

    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} "
        f"Accuracy: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, BEST_MODEL_PATH)
        print("Best model saved.")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(best_model_state)

Epoch [1/25] Loss: 2.4938 Accuracy: 0.3423 Val Loss: 2.3223 Val Accuracy: 0.3744
Best model saved.
Epoch [2/25] Loss: 1.4886 Accuracy: 0.6044 Val Loss: 2.1772 Val Accuracy: 0.4154
Best model saved.
Epoch [3/25] Loss: 1.2011 Accuracy: 0.6637 Val Loss: 2.2172 Val Accuracy: 0.3564
Epoch [4/25] Loss: 1.0308 Accuracy: 0.7242 Val Loss: 2.2365 Val Accuracy: 0.4179
Best model saved.
Epoch [5/25] Loss: 0.9137 Accuracy: 0.7423 Val Loss: 2.3400 Val Accuracy: 0.4333
Best model saved.
Epoch [6/25] Loss: 0.8282 Accuracy: 0.7588 Val Loss: 2.3996 Val Accuracy: 0.4385
Best model saved.
Epoch [7/25] Loss: 0.7452 Accuracy: 0.7901 Val Loss: 2.4840 Val Accuracy: 0.4000
Epoch [8/25] Loss: 0.6987 Accuracy: 0.7978 Val Loss: 2.6220 Val Accuracy: 0.4410
Best model saved.
Epoch [9/25] Loss: 0.6642 Accuracy: 0.8055 Val Loss: 2.6950 Val Accuracy: 0.3872
Epoch [10/25] Loss: 0.6348 Accuracy: 0.8165 Val Loss: 2.7213 Val Accuracy: 0.3949
Epoch [11/25] Loss: 0.5923 Accuracy: 0.8214 Val Loss: 2.8644 Val Accuracy: 0.4205

<All keys matched successfully>

In [12]:
torch.save(
    model.state_dict(),
    "models/efficientnetb0_final.pth"
)

print("\nModel Saved Successfully")


Model Saved Successfully
